In [1]:
import numpy as np
import pandas as pd
from sklearn import metrics
import os

In [2]:
pu_labels = pd.read_csv("../data/pu_labels.csv", index_col=0)

In [3]:
def get_validation_preds(scoring_name="svc_5", clustering_name="pumkl", scoring_column="Probability of pseudo-positivity", category="max", exclude=False):
    train_targets, test_targets = [], []
    train_pred, test_pred = [], []
    scoring_model = scoring_name.split("_")[0]
    if exclude:
        validation_folder = category+"_exclude"
    else:
        validation_folder = category+"_include"
    root = os.path.join("..","validation",validation_folder,"scoring",scoring_name,clustering_name)
    if category=="max":
        n=16
    else:
        n=10
    for i in range(n):
        path = os.path.join(root, f"{scoring_model}_{clustering_name}_150_scores_val{i}.xlsx")
        validation_predictions = pd.read_excel(path, sheet_name=0)

        pseudo_targets = (validation_predictions["Original cluster"]>0).to_numpy().astype(int)
        predictions = validation_predictions[scoring_column].to_numpy()

        train_indices = np.arange(len(pu_labels))
        if category=="max":
            test_indices = np.nonzero(pu_labels["class"])[0][i:i+1]
        else:
            test_indices = np.arange(len(pu_labels))[pu_labels["class"]==i+1]
        train_indices = np.setxor1d(train_indices, test_indices)

        train_targets += [pseudo_targets[train_indices]]
        test_targets += [pseudo_targets[test_indices]]

        train_pred += [predictions[train_indices]]
        test_pred += [predictions[test_indices]]

    return (train_targets, test_targets), (train_pred, test_pred)
    

# MAX-include validation

In [4]:
def compute_max_include_scores(train_targets, test_targets, train_pred, test_pred):
    n = len(train_targets)
    f1_scores, accuracy_scores, auc_scores = np.zeros(n), np.zeros(n), np.zeros(n)
    convertability_scores, predicted_positive = np.zeros(n), np.zeros(n)
    for i in range(n):
        all_targets = np.concatenate([train_targets[i], test_targets[i]])
        all_preds = np.concatenate([train_pred[i], test_pred[i]])
        
        f1_scores[i] = metrics.f1_score(all_targets, all_preds>0.5)
        accuracy_scores[i] = metrics.accuracy_score(all_targets, all_preds>0.5)
        auc_scores[i] = metrics.roc_auc_score(all_targets, all_preds)

        predicted_positive[i] = test_targets[i].item()
        convertability_scores[i] = test_pred[i].item()

    print(f"\tF1: {np.mean(f1_scores):.3f} +/- {np.std(f1_scores):.3f}")
    print(f"\tAccuracy: {np.mean(accuracy_scores):.3f} +/- {np.std(accuracy_scores):.3f}")
    print(f"\tAUC: {np.mean(auc_scores):.3f} +/- {np.std(auc_scores):.3f}")
    print(f"\tAverage score: {np.mean(convertability_scores):.3f} +/- {np.std(convertability_scores):.3f}")
    print(f"\t\tWhen positive:  {np.mean(convertability_scores[predicted_positive==1]):.3f} +/- {np.std(convertability_scores[predicted_positive==1]):.3f}")
    print(f"\t\tWhen negative:  {np.mean(convertability_scores[predicted_positive==0]):.3f} +/- {np.std(convertability_scores[predicted_positive==0]):.3f}")

In [5]:
for clustering_model in ["kmeans", "copkmeans", "pckmeans", "pumkl"]:
    print(f"Scores for interpreting {clustering_model} with an svc")
    targets, preds = get_validation_preds("svc_5", clustering_model, scoring_column="Total Score", category="max", exclude=False)
    compute_max_include_scores(*targets, *preds)

Scores for interpreting kmeans with an svc
	F1: 0.605 +/- 0.131
	Accuracy: 0.962 +/- 0.010
	AUC: 0.989 +/- 0.003
	Average score: 0.692 +/- 0.262
		When positive:  0.738 +/- 0.200
		When negative:  0.009 +/- 0.000
Scores for interpreting copkmeans with an svc
	F1: 0.440 +/- 0.124
	Accuracy: 0.992 +/- 0.002
	AUC: 0.995 +/- 0.007
	Average score: 0.326 +/- 0.320
		When positive:  0.864 +/- 0.075
		When negative:  0.202 +/- 0.206
Scores for interpreting pckmeans with an svc
	F1: 0.426 +/- 0.123
	Accuracy: 0.952 +/- 0.012
	AUC: 0.982 +/- 0.008
	Average score: 0.563 +/- 0.267
		When positive:  0.594 +/- 0.227
		When negative:  0.428 +/- 0.366
Scores for interpreting pumkl with an svc
	F1: 0.932 +/- 0.012
	Accuracy: 0.987 +/- 0.003
	AUC: 0.997 +/- 0.001
	Average score: 0.716 +/- 0.328
		When positive:  0.855 +/- 0.140
		When negative:  0.536 +/- 0.405


# MXenes-include validation

In [6]:
def compute_mxene_include_scores(train_targets, test_targets, train_pred, test_pred):
    n = len(train_targets)
    f1_scores, accuracy_scores, auc_scores = np.zeros(n), np.zeros(n), np.zeros(n)
    for i in range(n):
        all_targets = np.concatenate([train_targets[i], test_targets[i]])
        all_preds = np.concatenate([train_pred[i], test_pred[i]])
        
        f1_scores[i] = metrics.f1_score(all_targets, all_preds>0.5)
        accuracy_scores[i] = metrics.accuracy_score(all_targets, all_preds>0.5)
        auc_scores[i] = metrics.roc_auc_score(all_targets, all_preds)

    print(f"\tF1: {np.mean(f1_scores):.3f} +/- {np.std(f1_scores):.3f}")
    print(f"\tAccuracy: {np.mean(accuracy_scores):.3f} +/- {np.std(accuracy_scores):.3f}")
    print(f"\tAUC: {np.mean(auc_scores):.3f} +/- {np.std(auc_scores):.3f}")

In [7]:
for clustering_model in ["kmeans", "copkmeans", "pckmeans", "pumkl"]:
    print(f"Scores for interpreting {clustering_model} with an svc")
    targets, preds = get_validation_preds("svc_5", clustering_model, scoring_column="Total Score", category="mx", exclude=False)
    compute_mxene_include_scores(*targets, *preds)

Scores for interpreting kmeans with an svc
	F1: 0.575 +/- 0.106
	Accuracy: 0.961 +/- 0.009
	AUC: 0.987 +/- 0.004
Scores for interpreting copkmeans with an svc
	F1: 0.458 +/- 0.124
	Accuracy: 0.990 +/- 0.003
	AUC: 0.997 +/- 0.001
Scores for interpreting pckmeans with an svc
	F1: 0.540 +/- 0.116
	Accuracy: 0.961 +/- 0.007
	AUC: 0.986 +/- 0.005
Scores for interpreting pumkl with an svc
	F1: 0.942 +/- 0.009
	Accuracy: 0.988 +/- 0.002
	AUC: 0.997 +/- 0.001
